# C-SWAP-разметка + случайные попарные причинные тесты на ResNet-18 (GPU, Colab)

Переработка `synergy_experiments/resnet_cswap_profile.py` для запуска на GPU в Colab -
локально на CPU один слой (layer1, 10 классов, 200 пар/категорию) считался >2 часа и
не был закончен, поэтому расчёт переносится сюда.

**Что делает ноутбук:**
1. Берёт ResNet-18 (обученную на ImageNet и случайно инициализированную).
2. Размечает каналы каждого слоя как `critical` / `harmful` ("detrimental" в терминах
   статьи) / `neutral` **по определению C-SWAP** (Bauvin, Baret, Ahmad; BMVC 2025,
   arXiv:2510.18636, Definition 1-2): scoring function = softmax-вероятность истинного
   класса, относительный эффект `xi_n`, усреднённый по всем классам, значимость -
   голосование (OR) по классам (paired t-test на каждый класс отдельно) + FDR-BY
   поправка и порог эффекта `tau` (наше дополнение сверх "голого" alpha=0.05 статьи).
3. На каждом слое для 4 направлений (`neutral-neutral`, `critical-neutral`,
   `harmful-neutral`, `harmful-critical`) берёт `N_PAIRS` РАВНОМЕРНО СЛУЧАЙНЫХ пар
   (без гессиан-скрининга - он признан неприменимым для ReLU+BatchNorm(.eval), см.
   `NOVELTY_AND_FINDINGS.md` раздел 3.2) и тестирует точный причинный $S(A,B) =
   TE(A)+TE(B)-TE(A,B)$, $TE = \text{baseline\_logit} - \text{ablated\_logit}$, с
   FDR-BY коррекцией и порогом $\tau$ - тот же протокол, что и во всех остальных
   экспериментах проекта.
4. Повторяет для обученной и необученной (случайная инициализация + калибровка
   BatchNorm) сети.
5. Сохраняет чекпойнт после каждого слоя (на случай обрыва сессии Colab) и строит
   итоговую таблицу слой x категория: сколько пар протестировано, сколько
   положительных/отрицательных значимых, какой процент всех пар покрыт.

**Важно про масштаб:** даже на GPU полный прогон (4 слоя x 2 сети x 10 классов x
200 пар x 4 категории) может занимать заметное время на раннем слое (`layer1`) из-за
длинного "хвоста" сети, досчитываемого от каждого варианта занулённого канала/пары до
конца сети. Чекпойнты позволяют не терять прогресс.


## 1. Установка зависимостей и проверка GPU

В Colab выберите Runtime -> Change runtime type -> GPU, прежде чем выполнять ячейки.


In [1]:
!pip install -q statsmodels

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


torch: 2.11.0+cu130 | CUDA available: True
GPU: Tesla T4


## 2. Google Drive + данные (10 классов ImageNet x 50 изображений)

Эта секция монтирует Google Drive (нужен и для данных, и для устойчивости чекпойнтов
к обрыву сессии - см. раздел 6) и готовит `DATA_ROOT`.

**Важно: `DATA_ROOT` - это ПУТЬ К ПАПКЕ** с 10 подпапками-классами (каждая с ~50
изображениями), **не путь к zip-файлу**. `ImageFolder` не умеет читать прямо из zip.
Если у вас в Drive лежит `imagenet_paper_val.zip`, путь к САМОМУ zip-файлу (со
расширением `.zip`) указывается в `DATA_SOURCE_PATH` ниже только для распаковки, а
`DATA_ROOT` ноутбук определит сам ПОСЛЕ распаковки (путь без `.zip`, это будет папка
в `/content`, а не в Drive).

Выберите `DATA_SOURCE`:
- `"upload"` - загрузить `imagenet_paper_val.zip` через диалог (не нужен Drive для данных).
- `"drive_zip"` - zip уже лежит в вашем Drive; укажите путь к нему в `DATA_SOURCE_PATH`
  (с `.zip` на конце - это путь к файлу).


In [2]:
# --- монтируем Drive (нужен для DATA_SOURCE="drive_zip" и для устойчивых чекпойнтов) ---
MOUNT_DRIVE = True  # поставьте False, если НЕ хотите использовать Drive вообще (тогда
                     # DATA_SOURCE обязан быть "upload", а чекпойнты останутся только в
                     # /content этой сессии - не переживут обрыв раннтайма, см. раздел 6)

import os
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')  # если уже смонтирован в этой сессии - просто напомнит об этом, не ошибка

# --- данные: выберите источник ---
DATA_SOURCE = "drive_zip"  # "upload" или "drive_zip"

# путь К САМОМУ ZIP-ФАЙЛУ (с расширением .zip) - используется только если DATA_SOURCE="drive_zip".
# Это путь к файлу, а НЕ к папке, в которую он распакуется - DATA_ROOT ниже будет другим.
DATA_SOURCE_PATH = "/content/drive/MyDrive/c_swap_experiment/imagenet_paper_val.zip"

import shutil
import zipfile

if DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()  # выберите imagenet_paper_val.zip
    zip_path = next(iter(uploaded.keys()))
elif DATA_SOURCE == "drive_zip":
    assert MOUNT_DRIVE, 'DATA_SOURCE="drive_zip" требует MOUNT_DRIVE=True'
    zip_path = DATA_SOURCE_PATH
    assert os.path.isfile(zip_path), f"не найден файл: {zip_path}"
else:
    raise ValueError(f'неизвестный DATA_SOURCE: {DATA_SOURCE!r} (ожидается "upload" или "drive_zip")')

# ВСЕГДА распаковываем заново (а не "если папки ещё нет") - иначе повторный запуск
# этой ячейки после прерванной/неудачной первой распаковки молча использует старую,
# возможно неполную, папку.
extract_dir = "/content/_unzipped"
if os.path.isdir(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    print(f"архив: {len(names)} записей, например: {names[:3]}")
    # Некоторые архиваторы (типичная причина - zip, собранный на Windows не через
    # Python/стандартный zip, а другим инструментом) хранят внутренние пути с
    # ОБРАТНЫМ слэшем "" вместо "/" (требуемого zip-форматом). zipfile.extractall на
    # Linux (Colab) не распознаёт "" как разделитель папок - такие записи высыпаются
    # ПЛОСКО, по одному файлу на запись, с буквальным "" внутри имени файла, а не как
    # вложенные папки. Нормализуем вручную, если видим такое.
    has_backslash = any("\\" in n for n in names)
    if has_backslash:
        print("Обнаружены обратные слэши во внутренних путях архива - распаковываю вручную с нормализацией")
        for info in zf.infolist():
            normalized = info.filename.replace("\\", "/")
            target = os.path.join(extract_dir, normalized)
            if normalized.endswith("/"):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with zf.open(info) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    else:
        zf.extractall(extract_dir)

# архив может содержать сами 10 папок классов прямо в корне, либо быть обёрнут в 1-2
# дополнительных уровня папок (напр. "val/", или "imagenet_paper_val/val/") - ищем
# НА ГЛУБИНУ ДО 2 УРОВНЕЙ директорию, где лежат >= 5 подпапок с изображениями
def find_class_root(base, max_depth=2):
    stack = [(base, 0)]
    while stack:
        cur, depth = stack.pop(0)
        if not os.path.isdir(cur):
            continue
        subdirs = [d for d in os.listdir(cur) if os.path.isdir(os.path.join(cur, d))]
        if len(subdirs) >= 5:
            return cur
        if depth < max_depth:
            for d in subdirs:
                stack.append((os.path.join(cur, d), depth + 1))
    return None


DATA_ROOT = find_class_root(extract_dir)
if DATA_ROOT is None:
    # диагностика вместо голого AssertionError - чтобы сразу было видно реальную структуру архива
    print("Не нашёл папку с >=5 подпапками-классами. Содержимое архива (до 2 уровней):")
    for root, dirs, filenames in os.walk(extract_dir):
        depth = root[len(extract_dir):].count(os.sep)
        if depth > 2:
            dirs[:] = []
            continue
        print(f"{'  ' * depth}{root}/  (папок: {len(dirs)}, файлов: {len(filenames)})")
    print("Примеры имён файлов (первые 5):",
          [f for f in os.listdir(extract_dir)[:5]])
    raise AssertionError(
        "не нашёл папку с >=5 подпапками-классами внутри архива - смотри структуру и "
        "примеры имён файлов выше. Если в именах виден буквальный символ '\\' - значит "
        "нормализация выше не сработала (напр. двойной обратный слэш в архиве) - "
        "перезалейте архив как обычный zip с прямыми слэшами (создан на Linux/Mac, "
        "или через Python shutil.make_archive, а не стандартным Windows-архиватором)."
    )

print("DATA_ROOT =", DATA_ROOT, "(папка, НЕ zip)")
print("классы:", sorted(os.listdir(DATA_ROOT)))

# --- OUT_DIR для чекпойнтов (раздел 6) - тот же Drive, если смонтирован ---
if MOUNT_DRIVE:
    OUT_DIR = "/content/drive/MyDrive/resnet_cswap_profile"
    os.makedirs(OUT_DIR, exist_ok=True)
else:
    OUT_DIR = "/content"
print("OUT_DIR =", OUT_DIR, "(здесь будет лежать resnet_cswap_profile.json)")


Mounted at /content/drive
архив: 500 записей, например: ['n01440764/013862900598484.jpg', 'n01440764/029338166850799.jpg', 'n01440764/039237931249806.jpg']
DATA_ROOT = /content/_unzipped (папка, НЕ zip)
классы: ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475', 'n01496331', 'n01498041', 'n01514668', 'n01514859', 'n01518878']
OUT_DIR = /content/drive/MyDrive/resnet_cswap_profile (здесь будет лежать resnet_cswap_profile.json)


In [3]:
# (эта ячейка больше не нужна отдельно - монтирование Drive и OUT_DIR
# теперь входят в ячейку раздела 2 выше)


## 3. Конфигурация

Параметры масштаба согласованы заранее: 4 представительных слоя (как в
`resnet_layer_interactions.py`), все 10 классов ImageNet-10 субсета, 200 случайных
пар на слой x категорию. `OUT_DIR` берётся из раздела 2 выше (Drive, если
`MOUNT_DRIVE=True`, иначе `/content`) - выполните ячейки по порядку.

`RUN_CHUNK` ниже - это чанк ПО УМОЛЧАНИЮ для самого глубокого слоя (`layer4`, короткий
"хвост", маленькие активации 7x7). Для более ранних слоёв (`layer1`-`layer3`) чанк
масштабируется вниз автоматически в разделе 5 (`analyze_layer`) - см. пояснение там:
точка вмешательства в `layer1` имеет САМЫЙ ДЛИННЫЙ оставшийся "хвост" сети И САМОЕ
БОЛЬШОЕ пространственное разрешение активаций (56x56 против 7x7 у `layer4`), поэтому
тот же `RUN_CHUNK=64` там требует на порядок больше GPU-памяти и может уронить сессию
в `CUDA OutOfMemoryError`.


In [4]:
import json
import time
import itertools
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

SEED = 42
ALPHA = 0.05
FDR_METHOD = "fdr_by"      # Benjamini-Yekutieli: парные тесты структурно зависимы
TAU_REL = 0.02             # порог размера эффекта, как во всех остальных экспериментах проекта
N_CAL = 128                # калибровочных изображений на класс (на деле capped датасетом до 50)
N_CLASSES = 10             # весь текущий imagenet-10 субсет
N_PAIRS = 200              # случайных пар на слой x категорию
PROB_FLOOR = 1e-3          # см. ячейку 5 - защита от деления на почти-нулевую baseline-вероятность
RUN_CHUNK = 64             # чанк ДЛЯ САМОГО ГЛУБОКОГО слоя (layer4, короткий хвост, активации 7x7) -
                           # для более ранних слоёв реальный чанк МЕНЬШЕ, см. layer_chunk() в разделе 5
                           # (иначе падает в CUDA OOM на layer1/layer2 - длинный хвост + большие активации)

MEAN = (.485, .456, .406)
STD = (.229, .224, .225)
TRANSFORM = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

# (stage_idx 0-based, block_idx, n_channels, имя) - те же 4 представительных слоя,
# что в resnet_layer_interactions.py / resnet_layer_profile.json
LAYERS = [(0, 1, 64, "layer1.1.conv2"), (1, 1, 128, "layer2.1.conv2"),
          (2, 1, 256, "layer3.1.conv2"), (3, 1, 512, "layer4.1.conv2")]

# OUT_DIR задан в ячейке раздела 2 (Drive или /content) - здесь только путь к файлу
RESULTS_PATH = OUT_DIR + "/resnet_cswap_profile.json"
CATEGORIES = ["neutral_neutral", "critical_neutral", "harmful_neutral", "harmful_critical"]


## 4. Базовые примитивы (do-вмешательство на активациях)

Перенесено из `resnet_layer_interactions.py` без изменения причинного механизма -
только добавлено размещение тензоров на `device` (GPU/CPU).


In [5]:
def get_stages(model):
    return [model.layer1, model.layer2, model.layer3, model.layer4]


def capture_block_io(model, stage_idx, block_idx, x_batch):
    """Прогоняет x_batch один раз, попутно захватывая вход блока (для identity) и
    вход conv2 (a, 'префикс'). Возвращает (a, identity, baseline_logits)."""
    stages = get_stages(model)
    block = stages[stage_idx][block_idx]
    captured = {}

    def pre_block(module, inp):
        captured["x"] = inp[0].detach()

    def pre_conv2(module, inp):
        captured["a"] = inp[0].detach()

    h1 = block.register_forward_pre_hook(pre_block)
    h2 = block.conv2.register_forward_pre_hook(pre_conv2)
    with torch.no_grad():
        logits = model(x_batch)
    h1.remove()
    h2.remove()
    x_in = captured["x"]
    identity = block.downsample(x_in) if block.downsample is not None else x_in
    return captured["a"], identity, logits


def run_from(model, stage_idx, block_idx, a, identity):
    """Доcчитывает сеть от входа conv2 указанного блока до логитов: остаток текущего
    блока + все последующие блоки/стадии + avgpool + fc."""
    stages = get_stages(model)
    block = stages[stage_idx][block_idx]
    out = block.bn2(block.conv2(a))
    out = out + identity
    x = block.relu(out)
    for b in stages[stage_idx][block_idx + 1:]:
        idn = b.downsample(x) if b.downsample is not None else x
        out = b.relu(b.bn1(b.conv1(x)))
        out = b.bn2(b.conv2(out))
        x = b.relu(out + idn)
    for s in stages[stage_idx + 1:]:
        for b in s:
            idn = b.downsample(x) if b.downsample is not None else x
            out = b.relu(b.bn1(b.conv1(x)))
            out = b.bn2(b.conv2(out))
            x = b.relu(out + idn)
    x = model.avgpool(x)
    x = torch.flatten(x, 1)
    return model.fc(x)


def batched_variant_logits(model, stage_idx, block_idx, a, identity, variants, target, chunk=None):
    """variants: список кортежей номеров каналов для зануления (() = базовый, (i,) = одиночный,
    (i, j) = пара). Возвращает массив [len(variants), B] логитов целевого класса."""
    chunk = chunk or RUN_CHUNK
    B = a.shape[0]
    out_rows = []
    with torch.no_grad():
        for start in range(0, len(variants), chunk):
            batch_variants = variants[start:start + chunk]
            k = len(batch_variants)
            a_rep = a.unsqueeze(0).repeat(k, 1, 1, 1, 1).clone()
            for vi, channels in enumerate(batch_variants):
                for c in channels:
                    a_rep[vi, :, c, :, :] = 0.0
            id_rep = identity.unsqueeze(0).repeat(k, 1, 1, 1, 1)
            a_flat = a_rep.view(k * B, *a.shape[1:])
            id_flat = id_rep.reshape(k * B, *identity.shape[1:])
            logits = run_from(model, stage_idx, block_idx, a_flat, id_flat)
            logits = logits.view(k, B, -1)[:, :, target].cpu().numpy()
            out_rows.append(logits)
    return np.concatenate(out_rows, axis=0)


def batched_variant_full_logits(model, stage_idx, block_idx, a, identity, variants, chunk=None):
    """Как batched_variant_logits, но возвращает ПОЛНЫЙ вектор логитов (без среза по
    target) - нужен для softmax-вероятности истинного класса (C-SWAP)."""
    chunk = chunk or RUN_CHUNK
    B = a.shape[0]
    out_rows = []
    with torch.no_grad():
        for start in range(0, len(variants), chunk):
            batch_variants = variants[start:start + chunk]
            k = len(batch_variants)
            a_rep = a.unsqueeze(0).repeat(k, 1, 1, 1, 1).clone()
            for vi, channels in enumerate(batch_variants):
                for c in channels:
                    a_rep[vi, :, c, :, :] = 0.0
            id_rep = identity.unsqueeze(0).repeat(k, 1, 1, 1, 1)
            a_flat = a_rep.view(k * B, *a.shape[1:])
            id_flat = id_rep.reshape(k * B, *identity.shape[1:])
            logits = run_from(model, stage_idx, block_idx, a_flat, id_flat)
            out_rows.append(logits.view(k, B, -1).cpu().numpy())
    return np.concatenate(out_rows, axis=0)


def softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def mean_test(values, tau):
    _, p = stats.ttest_1samp(values, 0.0)
    return {"ate": float(np.mean(values)), "p": float(np.nan_to_num(p, nan=1.0))}


def fdr(rows, alpha, method):
    if not rows:
        return rows
    reject, p_adj, _, _ = multipletests([r["p"] for r in rows], alpha=alpha, method=method)
    for r, ok, p in zip(rows, reject, p_adj):
        r["p_fdr"], r["significant"] = float(p), bool(ok)
    return rows


def calibrate_batchnorm(model, dataset, n_images=256, batch_size=32, seed=SEED):
    """У свежей (необученной) сети running_mean=0/running_var=1 - заводские заглушки.
    Прогоняем сеть в .train() на обычных изображениях без вмешательств один раз,
    затем .eval() - те же реальные (хоть и не оптимизированные под задачу) статистики
    BatchNorm для всех дальнейших вариантов, как у обученной сети."""
    model.train()
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(dataset), size=min(n_images, len(dataset)), replace=False)
    with torch.no_grad():
        for start in range(0, len(idxs), batch_size):
            batch_idx = idxs[start:start + batch_size]
            xb = torch.stack([dataset[i][0] for i in batch_idx]).to(device)
            model(xb)
    model.eval()


def calib_batch(dataset, class_idx, n, seed):
    idxs = [i for i, (_, y) in enumerate(dataset.samples) if y == class_idx]
    rng = np.random.default_rng(seed + class_idx)
    pick = rng.choice(idxs, size=min(n, len(idxs)), replace=False)
    xs = torch.stack([dataset[i][0] for i in pick]).to(device)
    return xs


def sanity_check(model, dataset):
    """run_from(model, ..., a, identity) без изменений должен воспроизвести исходные
    логиты - проверка эквивалентности доcчёта 'хвоста' полному forward."""
    x = calib_batch(dataset, 0, 4, SEED)
    for stage_idx, block_idx, _, name in LAYERS:
        a, identity, baseline_logits = capture_block_io(model, stage_idx, block_idx, x)
        recon = run_from(model, stage_idx, block_idx, a, identity)
        diff = (recon - baseline_logits).abs().max().item()
        assert diff < 1e-4, f"{name}: recon diff {diff} too large"
        print(f"  sanity ok: {name}, max|diff|={diff:.2e}")


## 5. Разметка каналов по C-SWAP (Definition 1-2) + случайный отбор пар

`PROB_FLOOR` - численная защита: C-SWAP-овский относительный эффект
$(\sigma^*-\sigma)/\sigma$ делится на baseline-вероятность истинного класса; если для
калибровочного изображения сеть и без вмешательства уже почти не уверена в истинном
классе ($\sigma(x)\approx 0$), деление взрывается (на одном изображении обученной сети
наблюдался относительный эффект ~2.17 при typical значениях ~0.02-0.07) и задирает
дисперсию канала так, что тест теряет значимость сразу для ВСЕХ каналов слоя.

Раньше такие изображения ИСКЛЮЧАЛИСЬ из калибровочного батча (`baseline_prob >
PROB_FLOOR`). Это ломалось на необученной (случайно инициализированной) сети: для
случайной сети softmax-вероятность ЛЮБОГО класса в среднем ~1/1000 = 0.001 - то есть
сам типичный, не патологический уровень уверенности случайной сети уже лежит у порога
`PROB_FLOOR=1e-3`. Для некоторых классов случайно ВСЕ калибровочные изображения
оказывались ниже порога, батч исключения пустел целиком (`B=0`), и `.view(k, 0, -1)`
падал с `RuntimeError: cannot reshape tensor of 0 elements` - воспроизведено на
`layer1.1.conv2` необученной сети.

Исправлено: вместо исключения изображений теперь **клипуется знаменатель** -
`denom = max(sigma(x), PROB_FLOOR)`, а не `sigma(x)` на прямую. Изображение никогда не
выбрасывается (батч всегда полного размера, `B=0` невозможен в принципе), а вклад
изображений с почти нулевой `sigma(x)` просто ограничен сверху `1/PROB_FLOOR`, вместо
того чтобы устремляться к бесконечности.

`layer_chunk(n_channels)` - адаптивный размер чанка (число вариантов занулённых
каналов/пар, обрабатываемых одним forward-проходом "хвоста" сети). Нужен из-за
`CUDA OutOfMemoryError`, воспроизведённого на `layer1.1.conv2` с константным
`RUN_CHUNK=64`: точка вмешательства в `layer1` - это вход в ПОСЛЕДНИЙ блок стадии, то
есть "хвост" включает почти всю остальную сеть (весь layer2+layer3+layer4), и сразу
после вмешательства активации ещё крупные (56x56x64), тогда как в `layer4` хвост -
только остаток последнего блока (активации уже 7x7x512). Чанк=64 на `layer1` означает
batch = 64 x (число калибровочных изображений) через ПОЧТИ ВСЮ сеть при 56x56 -
на T4 (~15 ГБ) это надёжно валит `CUDA OutOfMemoryError`. Формула `n_channels // 8`
масштабирует чанк вниз пропорционально тому, насколько канал "ранний" (у ResNet-18
число каналов на стадии обратно пропорционально оставшейся длине хвоста и площади
активаций), давая примерно одинаковый пиковый расход памяти на всех 4 слоях.


In [6]:
def layer_chunk(n_channels):
    """Адаптивный размер чанка по числу каналов слоя - см. пояснение в разделе 5.
    Для LAYERS проекта (64/128/256/512) даёт 8/16/32/64 - то же RUN_CHUNK=64, что и
    раньше, но ТОЛЬКО для layer4 (короткий хвост, активации 7x7); для более ранних
    слоёв пропорционально меньше, чтобы не словить CUDA OOM."""
    return max(4, min(RUN_CHUNK, n_channels // 8))


def label_channels_cswap(model, stage_idx, block_idx, n_channels, dataset, classes, layer_name="", chunk=None):
    """Роль канала по Definition 1-2 C-SWAP: sigma(x) = softmax-вероятность истинного
    класса; xi_n = mean[(sigma*(x)-sigma(x))/sigma(x)], усреднено по ВСЕМ классам сразу;
    значимость - paired t-test ОТДЕЛЬНО по каждому классу (голосование OR) + FDR-BY и
    порог tau (наше дополнение сверх alpha=0.05 статьи). Возвращает (roles, per_class_te,
    per_class_logit_std, cache) - te/cache нужны дальше для причинного теста S."""
    chunk = chunk or RUN_CHUNK
    per_class_te, per_class_rel = {}, {}
    per_class_logit_std, per_class_prob_std = {}, {}
    cache = {}
    for class_idx in tqdm(classes, desc=f"{layer_name}: разметка по классам", leave=False):
        x = calib_batch(dataset, class_idx, N_CAL, SEED)
        a, identity, baseline_full = capture_block_io(model, stage_idx, block_idx, x)
        baseline_full_np = baseline_full.cpu().numpy()
        baseline_logit = baseline_full_np[:, class_idx]
        baseline_prob = softmax_np(baseline_full_np)[:, class_idx]

        variants = [(c,) for c in range(n_channels)]
        full_single = batched_variant_full_logits(model, stage_idx, block_idx, a, identity, variants, chunk=chunk)
        logit_single = full_single[:, :, class_idx]
        prob_single = softmax_np(full_single)[:, :, class_idx]

        te_single = baseline_logit[None, :] - logit_single
        # клип знаменателя вместо исключения изображений (см. раздел 5) - денominатор
        # никогда не опускается ниже PROB_FLOOR, поэтому батч никогда не пустеет
        denom = np.maximum(baseline_prob[None, :], PROB_FLOOR)
        rel_single = (prob_single - baseline_prob[None, :]) / denom

        per_class_te[class_idx] = te_single
        per_class_rel[class_idx] = rel_single
        per_class_logit_std[class_idx] = float(np.std(baseline_logit, ddof=1))
        per_class_prob_std[class_idx] = float(np.std(baseline_prob, ddof=1))
        cache[class_idx] = (a, identity, baseline_logit)

    sig_per_class = {}
    for class_idx in classes:
        tau = TAU_REL * per_class_prob_std[class_idx]
        rows = []
        for c in range(n_channels):
            r = mean_test(per_class_rel[class_idx][c], tau)
            r["channel"] = c
            rows.append(r)
        fdr(rows, ALPHA, FDR_METHOD)
        sig_per_class[class_idx] = {r["channel"]: (r["significant"] and abs(r["ate"]) > tau) for r in rows}

    roles = {}
    for c in range(n_channels):
        pooled = np.concatenate([per_class_rel[k][c] for k in classes])
        xi = float(pooled.mean())
        any_sig = any(sig_per_class[k][c] for k in classes)
        if not any_sig:
            roles[c] = "neutral"
        elif xi <= 0:
            roles[c] = "critical"
        else:
            roles[c] = "harmful"  # "detrimental" в терминах статьи C-SWAP

    return roles, per_class_te, per_class_logit_std, cache


def sample_pairs(group_a, group_b, n, rng, same_group):
    """Равномерно случайные пары (без гессиан-ранжирования)."""
    if same_group:
        pool = list(itertools.combinations(sorted(set(group_a)), 2))
    else:
        ga, gb = sorted(set(group_a)), sorted(set(group_b))
        pool = sorted({(min(i, j), max(i, j)) for i in ga for j in gb if i != j})
    total = len(pool)
    if total == 0:
        return [], 0
    if total <= n:
        return pool, total
    idx = rng.choice(total, size=n, replace=False)
    return [pool[i] for i in idx], total


def test_pairs_for_class(model, stage_idx, block_idx, a, identity, class_idx, baseline_logit,
                          te_cache, tau, pairs, chunk=None):
    """Точный причинный тест S = TE(A)+TE(B)-TE(A,B) для заданного списка пар."""
    chunk = chunk or RUN_CHUNK
    if not pairs:
        return {"n_pairs_tested": 0, "n_syn": 0, "n_sub": 0}
    logits = batched_variant_logits(model, stage_idx, block_idx, a, identity, pairs, class_idx, chunk=chunk)
    rows = []
    for k, (i, j) in enumerate(pairs):
        te_joint = baseline_logit[None, :] - logits[k:k + 1]
        s_vals = te_cache[i] + te_cache[j] - te_joint[0]
        r = mean_test(s_vals, tau)
        r.update(a=i, b=j, S=r["ate"])
        rows.append(r)
    fdr(rows, ALPHA, FDR_METHOD)
    for r in rows:
        ok = r["significant"] and abs(r["S"]) > tau
        r["sign"] = "synergy" if ok and r["S"] > 0 else "subadditive" if ok and r["S"] < 0 else "additive"
    return {"n_pairs_tested": len(rows),
            "n_syn": sum(r["sign"] == "synergy" for r in rows),
            "n_sub": sum(r["sign"] == "subadditive" for r in rows)}


def analyze_layer(model, dataset, stage_idx, block_idx, n_channels, classes, seed, layer_name=""):
    chunk = layer_chunk(n_channels)
    roles, per_class_te, per_class_logit_std, cache = label_channels_cswap(
        model, stage_idx, block_idx, n_channels, dataset, classes, layer_name=layer_name, chunk=chunk)

    critical = [c for c, r in roles.items() if r == "critical"]
    harmful = [c for c, r in roles.items() if r == "harmful"]
    neutral = [c for c, r in roles.items() if r == "neutral"]

    groups = {"neutral_neutral": (neutral, neutral, True),
              "critical_neutral": (critical, neutral, False),
              "harmful_neutral": (harmful, neutral, False),
              "harmful_critical": (harmful, critical, False)}

    rng = np.random.default_rng(seed)
    cat_out = {}
    for cat in tqdm(CATEGORIES, desc=f"{layer_name}: категории пар", leave=False):
        ga, gb, same = groups[cat]
        pairs, total_possible = sample_pairs(ga, gb, N_PAIRS, rng, same)
        tested_list, syn_list, sub_list = [], [], []
        for class_idx in classes:
            a, identity, baseline_logit = cache[class_idx]
            tau = TAU_REL * per_class_logit_std[class_idx]
            res = test_pairs_for_class(model, stage_idx, block_idx, a, identity, class_idx,
                                        baseline_logit, per_class_te[class_idx], tau, pairs, chunk=chunk)
            tested_list.append(res["n_pairs_tested"])
            syn_list.append(res["n_syn"])
            sub_list.append(res["n_sub"])
        coverage = 100.0 * (len(pairs) / total_possible) if total_possible else 0.0
        cat_out[cat] = {"group_a_size": len(ga), "group_b_size": len(gb),
                         "total_possible_pairs": total_possible,
                         "n_pairs_sampled": len(pairs),
                         "coverage_pct": coverage,
                         "n_pairs_tested_mean": float(np.mean(tested_list)),
                         "n_syn_mean": float(np.mean(syn_list)),
                         "n_sub_mean": float(np.mean(sub_list))}

    return {"n_channels": n_channels, "n_critical": len(critical), "n_harmful": len(harmful),
            "n_neutral": len(neutral), "categories": cat_out}


## 6. Запуск: обученная и необученная сеть, чекпойнт + резюме после обрыва

Чекпойнт пишется в `RESULTS_PATH` после **каждого слоя** (не после всего прогона).
При старте ячейка ниже сначала проверяет, есть ли уже `RESULTS_PATH` (напр. вы
домонтировали тот же Drive после обрыва сессии) - если да, уже посчитанные пары
(`сеть`, `слой`) пропускаются, досчитываются только оставшиеся. Если обрыв случился
посреди слоя, этот слой целиком пересчитывается заново (чекпойнт гранулярен по
слоям, не по классам/парам внутри слоя).

Это реально защищает от потери прогресса, только если `OUT_DIR` указывает на Drive
(раздел 2, `MOUNT_DRIVE=True`) - при `OUT_DIR=/content` чекпойнт переживёт сбой
ячейки, но не переживёт обрыв/перезапуск самого раннтайма.


In [7]:
def save_checkpoint(rows, classes):
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump({"config": {"N_CAL": N_CAL, "N_CLASSES": N_CLASSES, "classes": classes,
                               "N_PAIRS": N_PAIRS, "ALPHA": ALPHA, "TAU_REL": TAU_REL,
                               "FDR_METHOD": FDR_METHOD, "PROB_FLOOR": PROB_FLOOR},
                   "rows": rows}, f, indent=2, ensure_ascii=False)
    print(f"  [чекпойнт сохранён -> {RESULTS_PATH}]")


def run_for_model(model, dataset, classes, label, rows, done):
    for stage_idx, block_idx, n_channels, name in LAYERS:
        if (label, name) in done:
            print(f"[{label}] {name}: уже есть в чекпойнте, пропускаю")
            continue
        t0 = time.perf_counter()
        res = analyze_layer(model, dataset, stage_idx, block_idx, n_channels, classes, SEED, layer_name=name)
        row = {"net_type": label, "layer": name, "n_channels": n_channels,
               "n_critical": res["n_critical"], "n_harmful": res["n_harmful"],
               "n_neutral": res["n_neutral"], "categories": res["categories"],
               "seconds": time.perf_counter() - t0}
        print(f"[{label}] {name}: critical={res['n_critical']}/{n_channels}, "
              f"harmful={res['n_harmful']}/{n_channels}, neutral={res['n_neutral']}/{n_channels} "
              f"({row['seconds']:.1f} s)")
        for cat in CATEGORIES:
            c = res["categories"][cat]
            print(f"    {cat}: syn={c['n_syn_mean']:.1f} sub={c['n_sub_mean']:.1f} "
                  f"/{c['n_pairs_tested_mean']:.0f} tested, coverage={c['coverage_pct']:.2f}% "
                  f"(groups {c['group_a_size']}x{c['group_b_size']}, total_possible={c['total_possible_pairs']})")
        rows.append(row)
        done.add((label, name))
        save_checkpoint(rows, classes)
    return rows


torch.manual_seed(SEED)
dataset = datasets.ImageFolder(DATA_ROOT, transform=TRANSFORM)
classes = list(range(N_CLASSES))
print(f"dataset: {len(dataset)} images, классы {classes} из {sorted(dataset.classes)}")

# --- резюме: если чекпойнт уже существует (напр. после обрыва и повторного монтирования
# того же Drive-пути), подхватываем его и пропускаем уже посчитанные (сеть, слой) ---
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        rows = json.load(f)["rows"]
    print(f"Найден чекпойнт: {len(rows)} строк(и), продолжаю с того места, где остановились")
else:
    rows = []
done = {(r["net_type"], r["layer"]) for r in rows}

print("--- sanity check (обученная сеть) ---")
model_trained = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
sanity_check(model_trained, dataset)

print("\n--- обученная ResNet-18 (разметка C-SWAP) ---")
run_for_model(model_trained, dataset, classes, "обученная", rows, done)

print("\n--- необученная (случайная инициализация) ResNet-18 ---")
model_untrained = models.resnet18(weights=None).to(device)
calibrate_batchnorm(model_untrained, dataset)
model_untrained.eval()
sanity_check(model_untrained, dataset)
run_for_model(model_untrained, dataset, classes, "необученная", rows, done)

print("\nГотово.", RESULTS_PATH)


dataset: 500 images, классы [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] из ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475', 'n01496331', 'n01498041', 'n01514668', 'n01514859', 'n01518878']
Найден чекпойнт: 4 строк(и), продолжаю с того места, где остановились
--- sanity check (обученная сеть) ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 72.0MB/s]


  sanity ok: layer1.1.conv2, max|diff|=0.00e+00
  sanity ok: layer2.1.conv2, max|diff|=0.00e+00
  sanity ok: layer3.1.conv2, max|diff|=0.00e+00
  sanity ok: layer4.1.conv2, max|diff|=0.00e+00

--- обученная ResNet-18 (разметка C-SWAP) ---
[обученная] layer1.1.conv2: уже есть в чекпойнте, пропускаю
[обученная] layer2.1.conv2: уже есть в чекпойнте, пропускаю
[обученная] layer3.1.conv2: уже есть в чекпойнте, пропускаю
[обученная] layer4.1.conv2: уже есть в чекпойнте, пропускаю

--- необученная (случайная инициализация) ResNet-18 ---
  sanity ok: layer1.1.conv2, max|diff|=0.00e+00
  sanity ok: layer2.1.conv2, max|diff|=0.00e+00
  sanity ok: layer3.1.conv2, max|diff|=0.00e+00
  sanity ok: layer4.1.conv2, max|diff|=0.00e+00


layer1.1.conv2: разметка по классам:   0%|          | 0/10 [00:00<?, ?it/s]

layer1.1.conv2: категории пар:   0%|          | 0/4 [00:00<?, ?it/s]

...

layer3.1.conv2: категории пар:   0%|          | 0/4 [00:00<?, ?it/s]

[необученная] layer3.1.conv2: critical=138/256, harmful=118/256, neutral=0/256 (53.2 s)
    neutral_neutral: syn=0.0 sub=0.0 /0 tested, coverage=0.00% (groups 0x0, total_possible=0)
    critical_neutral: syn=0.0 sub=0.0 /0 tested, coverage=0.00% (groups 138x0, total_possible=0)
    harmful_neutral: syn=0.0 sub=0.0 /0 tested, coverage=0.00% (groups 118x0, total_possible=0)
    harmful_critical: syn=0.6 sub=0.4 /200 tested, coverage=1.23% (groups 118x138, total_possible=16284)
  [чекпойнт сохранён -> /content/drive/MyDrive/resnet_cswap_profile/resnet_cswap_profile.json]


layer4.1.conv2: разметка по классам:   0%|          | 0/10 [00:00<?, ?it/s]

layer4.1.conv2: категории пар:   0%|          | 0/4 [00:00<?, ?it/s]

[необученная] layer4.1.conv2: critical=258/512, harmful=254/512, neutral=0/512 (24.4 s)
    neutral_neutral: syn=0.0 sub=0.0 /0 tested, coverage=0.00% (groups 0x0, total_possible=0)
    critical_neutral: syn=0.0 sub=0.0 /0 tested, coverage=0.00% (groups 258x0, total_possible=0)
    harmful_neutral: syn=0.0 sub=0.0 /0 tested, coverage=0.00% (groups 254x0, total_possible=0)
    harmful_critical: syn=0.0 sub=0.0 /200 tested, coverage=0.31% (groups 254x258, total_possible=65532)
  [чекпойнт сохранён -> /content/drive/MyDrive/resnet_cswap_profile/resnet_cswap_profile.json]

Готово. /content/drive/MyDrive/resnet_cswap_profile/resnet_cswap_profile.json


## 7. Итоговая таблица: слой x категория x сеть

Колонки: `n_critical/n_harmful/n_neutral` - размер ролевых групп на слое (глобально,
по голосованию C-SWAP); `pairs_tested` - сколько пар реально протестировано (в среднем
по 10 классам); `synergy(+)`/`subadditive(-)` - число значимых положительных/
отрицательных связей; `coverage_%` - доля протестированных пар от всех возможных пар
этой категории на этом слое.


In [8]:
import pandas as pd

with open(RESULTS_PATH, encoding="utf-8") as f:
    data = json.load(f)

table_rows = []
for row in data["rows"]:
    for cat, c in row["categories"].items():
        table_rows.append({
            "net_type": row["net_type"], "layer": row["layer"], "category": cat,
            "n_critical": row["n_critical"], "n_harmful": row["n_harmful"], "n_neutral": row["n_neutral"],
            "pairs_tested": c["n_pairs_tested_mean"], "synergy(+)": c["n_syn_mean"],
            "subadditive(-)": c["n_sub_mean"], "coverage_%": round(c["coverage_pct"], 2),
            "group_a_size": c["group_a_size"], "group_b_size": c["group_b_size"],
            "total_possible_pairs": c["total_possible_pairs"],
        })
df = pd.DataFrame(table_rows)
df


,net_type,layer,category,n_critical,n_harmful,n_neutral,pairs_tested,synergy(+),subadditive(-),coverage_%,group_a_size,group_b_size,total_possible_pairs
0,обученная,layer1.1.conv2,neutral_neutral,6,3,55,200.0,0.6,1.2,13.47,55,55,1485
1,обученная,layer1.1.conv2,critical_neutral,6,3,55,200.0,0.3,0.1,60.61,6,55,330
2,обученная,layer1.1.conv2,harmful_neutral,6,3,55,165.0,0.2,0.2,100.00,3,55,165
3,обученная,layer1.1.conv2,harmful_critical,6,3,55,18.0,0.0,0.0,100.00,3,6,18
4,обученная,layer2.1.conv2,neutral_neutral,16,25,87,200.0,0.0,0.0,5.35,87,87,3741
5,обученная,layer2.1.conv2,critical_neutral,16,25,87,200.0,0.0,0.0,14.37,16,87,1392
6,обученная,layer2.1.conv2,harmful_neutral,16,25,87,200.0,0.0,0.0,9.20,25,87,2175
7,обученная,layer2.1.conv2,harmful_critical,16,25,87,200.0,0.0,0.0,50.00,25,16,400
8,обученная,layer3.1.conv2,neutral_neutral,23,20,213,200.0,0.0,0.0,0.89,213,213,22578
9,обученная,layer3.1.conv2,critical_neutral,23,20,213,200.0,0.0,0.0,4.08,23,213,4899


In [9]:
# Опционально: скачать итоговый JSON на локальный компьютер
from google.colab import files
files.download(RESULTS_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>